# 실시간 시각화 (Real-Time Visualization)

이 노트북은 시뮬레이션 결과의 실시간 시각화 및 인터랙티브 대시보드를 다룹니다.

## 목차
1. 실시간 애니메이션
2. 인터랙티브 파라미터 조정
3. 다중 플롯 대시보드
4. 성능 모니터링
5. 데이터 스트리밍

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'build'))

import _core as koo
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import time
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

koo.Logger.initialize("RealTimeViz")
koo.Logger.set_level(koo.LogLevel.INFO)
print(f"KooLab Version: {koo.version()}")

## 1. 실시간 1D 확산 애니메이션

In [ ]:
class Diffusion1D:
    def __init__(self, nx=100, L=1.0, D=0.01, dt=0.001):
        self.nx = nx
        self.L = L
        self.D = D
        self.dt = dt
        self.dx = L / (nx - 1)
        self.alpha = D * dt / (self.dx ** 2)
        
        # 초기 조건: 가우시안
        self.x = np.linspace(0, L, nx)
        self.C = np.exp(-((self.x - 0.5)**2) / (2 * 0.05**2))
        self.time = 0
        
        koo.Logger.info(f"1D Diffusion initialized: nx={nx}, CFL alpha={self.alpha:.4f}")
    
    def step(self):
        C_new = self.C.copy()
        for i in range(1, self.nx-1):
            C_new[i] = self.C[i] + self.alpha * (self.C[i+1] - 2*self.C[i] + self.C[i-1])
        self.C = C_new
        self.time += self.dt

# 애니메이션 생성
diffusion = Diffusion1D()

fig, ax = plt.subplots(figsize=(12, 6))
line, = ax.plot(diffusion.x, diffusion.C, 'b-', linewidth=2)
time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=14)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.1)
ax.set_xlabel('Position', fontsize=12)
ax.set_ylabel('Concentration', fontsize=12)
ax.set_title('1D Diffusion Animation', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

def animate(frame):
    for _ in range(10):  # 프레임당 10 스텝
        diffusion.step()
    line.set_ydata(diffusion.C)
    time_text.set_text(f'Time: {diffusion.time:.3f}s')
    return line, time_text

anim = FuncAnimation(fig, animate, frames=100, interval=50, blit=True)
plt.close()  # 정적 플롯 방지

koo.Logger.info("Animation created (run in Jupyter to see)")
HTML(anim.to_jshtml())

## 2. 2D 열 확산 애니메이션

In [ ]:
class HeatDiffusion2D:
    def __init__(self, size=50, D=0.1, dt=0.01):
        self.size = size
        self.D = D
        self.dt = dt
        
        # 초기 조건: 중앙에 뜨거운 점
        self.T = np.zeros((size, size))
        center = size // 2
        r = 5
        y, x = np.ogrid[:size, :size]
        mask = (x - center)**2 + (y - center)**2 <= r**2
        self.T[mask] = 1.0
        
        self.time = 0
    
    def laplacian(self, field):
        lapl = (
            np.roll(field, 1, axis=0) + 
            np.roll(field, -1, axis=0) + 
            np.roll(field, 1, axis=1) + 
            np.roll(field, -1, axis=1) - 
            4 * field
        )
        return lapl
    
    def step(self):
        dT = self.D * self.laplacian(self.T)
        self.T += dT * self.dt
        self.time += self.dt

heat = HeatDiffusion2D(size=50)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(heat.T, cmap='hot', vmin=0, vmax=1, interpolation='bilinear')
time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes, 
                    color='white', fontsize=14, fontweight='bold')
ax.set_title('2D Heat Diffusion', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Temperature')

def animate_heat(frame):
    for _ in range(5):
        heat.step()
    im.set_array(heat.T)
    time_text.set_text(f'Time: {heat.time:.2f}s')
    return [im, time_text]

anim2 = FuncAnimation(fig, animate_heat, frames=100, interval=50, blit=True)
plt.close()

koo.Logger.info("2D heat diffusion animation created")
HTML(anim2.to_jshtml())

## 3. 인터랙티브 파라미터 조정

ipywidgets를 사용하여 실시간으로 파라미터를 조정합니다.

In [ ]:
def interactive_diffusion(D=0.01, initial_width=0.05, nx=100):
    """인터랙티브 1D 확산 시뮬레이션"""
    L = 1.0
    dt = 0.001
    dx = L / (nx - 1)
    alpha = D * dt / (dx ** 2)
    
    # 초기 조건
    x = np.linspace(0, L, nx)
    C = np.exp(-((x - 0.5)**2) / (2 * initial_width**2))
    
    # 시뮬레이션
    C_final = C.copy()
    for _ in range(500):
        C_new = C_final.copy()
        for i in range(1, nx-1):
            C_new[i] = C_final[i] + alpha * (C_final[i+1] - 2*C_final[i] + C_final[i-1])
        C_final = C_new
    
    # 플롯
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(x, C, 'b-', linewidth=2, label='Initial', alpha=0.5)
    ax.plot(x, C_final, 'r-', linewidth=2, label='After 500 steps')
    ax.set_xlabel('Position', fontsize=12)
    ax.set_ylabel('Concentration', fontsize=12)
    ax.set_title(f'Interactive Diffusion (D={D}, width={initial_width}, alpha={alpha:.4f})', 
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.show()
    
    koo.Logger.info(f"Interactive plot: D={D}, width={initial_width}, CFL={alpha:.4f}")

# 인터랙티브 위젯
interact(interactive_diffusion,
         D=FloatSlider(min=0.001, max=0.05, step=0.001, value=0.01, description='Diffusion D:'),
         initial_width=FloatSlider(min=0.01, max=0.2, step=0.01, value=0.05, description='Width:'),
         nx=IntSlider(min=50, max=200, step=10, value=100, description='Grid points:'));

## 4. 다중 플롯 대시보드

In [ ]:
def create_dashboard(D=0.01, steps=1000):
    """종합 대시보드"""
    koo.Logger.info(f"Creating dashboard with D={D}, steps={steps}")
    
    # 시뮬레이션
    nx = 100
    L = 1.0
    dt = 0.001
    dx = L / (nx - 1)
    alpha = D * dt / (dx ** 2)
    
    x = np.linspace(0, L, nx)
    C = np.exp(-((x - 0.5)**2) / (2 * 0.05**2))
    
    # 데이터 수집
    C_history = [C.copy()]
    mass = [np.sum(C) * dx]
    max_C = [np.max(C)]
    time_points = [0]
    
    for step in range(1, steps+1):
        C_new = C.copy()
        for i in range(1, nx-1):
            C_new[i] = C[i] + alpha * (C[i+1] - 2*C[i] + C[i-1])
        C = C_new
        
        if step % 100 == 0:
            C_history.append(C.copy())
            mass.append(np.sum(C) * dx)
            max_C.append(np.max(C))
            time_points.append(step * dt)
    
    # 대시보드 생성
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    # 1. 농도 프로파일
    ax1 = fig.add_subplot(gs[0, :])
    for i, (C_snap, t) in enumerate(zip(C_history[::2], time_points[::2])):
        alpha_val = 0.3 + 0.7 * (i / len(C_history[::2]))
        ax1.plot(x, C_snap, alpha=alpha_val, linewidth=2, label=f't={t:.3f}s')
    ax1.set_xlabel('Position', fontsize=12)
    ax1.set_ylabel('Concentration', fontsize=12)
    ax1.set_title('Concentration Profiles', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper right', ncol=3, fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # 2. 질량 보존
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(time_points, mass, 'b-', linewidth=2)
    ax2.axhline(y=mass[0], color='r', linestyle='--', label='Initial mass')
    ax2.set_xlabel('Time (s)', fontsize=12)
    ax2.set_ylabel('Total Mass', fontsize=12)
    ax2.set_title('Mass Conservation', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. 최대 농도
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(time_points, max_C, 'r-', linewidth=2)
    ax3.set_xlabel('Time (s)', fontsize=12)
    ax3.set_ylabel('Max Concentration', fontsize=12)
    ax3.set_title('Peak Decay', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # 4. 최종 분포 히스토그램
    ax4 = fig.add_subplot(gs[2, 0])
    ax4.hist(C, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    ax4.set_xlabel('Concentration', fontsize=12)
    ax4.set_ylabel('Frequency', fontsize=12)
    ax4.set_title('Final Distribution Histogram', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='y')
    
    # 5. 통계 요약
    ax5 = fig.add_subplot(gs[2, 1])
    ax5.axis('off')
    stats_text = f"""
    Simulation Statistics
    ═══════════════════════
    
    Parameters:
      • Diffusion coeff (D): {D}
      • Time step (dt): {dt}
      • Grid spacing (dx): {dx:.4f}
      • CFL number (α): {alpha:.4f}
    
    Results:
      • Total steps: {steps}
      • Final time: {time_points[-1]:.3f}s
      • Initial mass: {mass[0]:.6f}
      • Final mass: {mass[-1]:.6f}
      • Mass change: {(mass[-1]-mass[0])/mass[0]*100:.4f}%
      • Initial max(C): {max_C[0]:.6f}
      • Final max(C): {max_C[-1]:.6f}
    """
    ax5.text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
             verticalalignment='center')
    
    plt.suptitle('Diffusion Simulation Dashboard', fontsize=18, fontweight='bold', y=0.995)
    plt.show()
    
    koo.Logger.info("Dashboard created successfully")

# 인터랙티브 대시보드
interact(create_dashboard,
         D=FloatSlider(min=0.001, max=0.05, step=0.001, value=0.01, description='D:'),
         steps=IntSlider(min=500, max=5000, step=500, value=2000, description='Steps:'));

## 5. 성능 모니터링

In [ ]:
def benchmark_diffusion(sizes=[50, 100, 200, 400], steps=1000):
    """다양한 그리드 크기에 대한 성능 벤치마크"""
    koo.Logger.info("Starting performance benchmark")
    
    timings = []
    
    for size in sizes:
        nx = size
        L = 1.0
        D = 0.01
        dt = 0.001
        dx = L / (nx - 1)
        alpha = D * dt / (dx ** 2)
        
        x = np.linspace(0, L, nx)
        C = np.exp(-((x - 0.5)**2) / (2 * 0.05**2))
        
        start = time.time()
        for _ in range(steps):
            C_new = C.copy()
            for i in range(1, nx-1):
                C_new[i] = C[i] + alpha * (C[i+1] - 2*C[i] + C[i-1])
            C = C_new
        elapsed = time.time() - start
        
        timings.append(elapsed)
        koo.Logger.info(f"Size {size}: {elapsed:.4f}s")
    
    # 플롯
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 실행 시간
    ax1.plot(sizes, timings, 'o-', linewidth=2, markersize=10, color='blue')
    ax1.set_xlabel('Grid Size', fontsize=12)
    ax1.set_ylabel('Execution Time (s)', fontsize=12)
    ax1.set_title('Performance vs Grid Size', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 초당 업데이트
    updates_per_sec = [steps / t for t in timings]
    ax2.plot(sizes, updates_per_sec, 's-', linewidth=2, markersize=10, color='green')
    ax2.set_xlabel('Grid Size', fontsize=12)
    ax2.set_ylabel('Updates per Second', fontsize=12)
    ax2.set_title('Throughput', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 요약 출력
    print("\nPerformance Summary:")
    print("="*60)
    for size, timing, ups in zip(sizes, timings, updates_per_sec):
        print(f"Size {size:4d}: {timing:8.4f}s  |  {ups:8.2f} updates/sec")
    print("="*60)
    
    koo.Logger.info("Benchmark completed")

benchmark_diffusion()

## 6. 요약

이 노트북에서 배운 내용:
- ✅ 실시간 애니메이션 생성 (1D, 2D)
- ✅ 인터랙티브 파라미터 조정 (ipywidgets)
- ✅ 다중 플롯 대시보드 구성
- ✅ 성능 벤치마크 및 모니터링
- ✅ 시뮬레이션 통계 수집 및 분석

### 주요 기능:
- FuncAnimation을 이용한 애니메이션
- ipywidgets를 이용한 실시간 파라미터 조정
- 질량 보존 및 피크 감쇠 모니터링
- 그리드 크기별 성능 비교

### 다음 단계:
- `04_gpu_acceleration.ipynb`: GPU 가속 및 성능 비교

In [ ]:
koo.Logger.info("Real-time visualization tutorial completed successfully!")